In [1]:
import sdmx
import time

/Users/nazimaniltepe/Projects/tuik-sdmx/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
sdmx.add_source({
    "id": "TUIK",
    "url": "https://nsiws.tuik.gov.tr/rest",
    "name": "Türkiye İstatistik Kurumu (TÜİK)"
}, override=True)

In [4]:
df = 'DF_YDUFE_EDO_V1'
tuik = sdmx.Client('TUIK')
print("START META", df)
start_m = time.perf_counter()
flow = tuik.dataflow(df, agency_id='TR', detail='full', references='descendants')
end_m = time.perf_counter()
print("END META", "elapsed", "%.3f"%(end_m-start_m), "sec")
dimensions = [comp.concept_identity.id 
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components]
measures = [msr.id for struct in flow.structure.items()
            for msr in struct[1].measures]
codelists = {comp.concept_identity.id: {k: v.name["tr"] for k, v in comp.local_representation.enumerated.items.items()}
            for struct in flow.structure.items()
            for comp in struct[1].dimensions.components if comp.local_representation.enumerated}
concepts = {concept.id: concept.name["tr"]
            for csch in flow.concept_scheme.values()
            for concept in csch.items.values() if concept.id in dimensions or concept.id in measures}
print("START SLEEP", "10 sec")
time.sleep(10)
print("END SLEEP")
print("START DATA", df)
start_d = time.perf_counter()
data = tuik.data(df)
end_d = time.perf_counter()
print("END DATA", "elapsed", "%.3f"%(end_d-start_d), "sec")
pd_data = sdmx.to_pandas(data)
for cl in codelists.keys():
    pd_data.index = pd_data.index.set_levels([codelists[vals.name][val] 
        for vals in pd_data.index.levels 
        for val in list(vals) if vals.name == cl
    ], level=cl)
pd_data.rename(concepts["OBS_VALUE"], inplace=True)
pd_data.index.set_names([concepts[name] for name in pd_data.index.names], inplace=True)
# pd_data.to_csv(f'/opt/airflow/files/{re.sub(r"^DF_", "", df)}.csv')

START META DF_YDUFE_EDO_V1
END META elapsed 5.509 sec
START SLEEP 10 sec
END SLEEP
START DATA DF_YDUFE_EDO_V1
END DATA elapsed 30.920 sec


KeyError: 'Level MAALIYET_GRUP not found'

In [16]:
len(data.data[0])

257065

In [65]:
len(data.data[0].obs)

257065

In [15]:
len(data.data[0].series)

1515

In [63]:
sum = 0
for serie, obs in data.data[0].series.items():
    sum += len(obs)

In [64]:
sum

257065

In [ ]:
{comp.concept_identity.id: {k: v.name["tr"] for k, v in comp.local_representation.enumerated.items.items()}
for struct in flow.structure.items()
for comp in struct[1].dimensions.components if comp.local_representation.enumerated}

In [11]:
codelists.keys()

dict_keys(['REF_AREA', 'INDICATOR', 'DEGISIM', 'BASE_PER', 'FREQ', 'FIYAT_ENDEKS_KAPSAM', 'MAALIYET_GRUP', 'FAALIYETCPA_2008', 'FAALIYET_NACE_REV2', 'FAALIYETCPA_2_1', 'FAAL_GRUP'])

In [4]:
pd_data

TIME_PERIOD  REF_AREA  INDICATOR                        DEGISIM                                    BASE_PER  FREQ   FIYAT_ENDEKS_KAPSAM  MALIYET_GRUP  FAALIYET_CPA_2008  FAALIYET_NACE_REV2  FAALIYET_CPA_2_1  FAAL_GRUP
2010-01      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             98.95
2010-02      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             96.26
2010-03      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                     2010      Aylık  Uygulanabilir değil  _Z            _Z                 B                   _Z                3             97.40
2010-04      Türkiye   Yurt Dışı Üretici Fiyat Endeksi  Endeks                                    

In [ ]:
pd_data

Zaman    Gözlem Sıklığı  Ölçü Birimi  Dış Ticaret Göstergeleri         Dış Ticaret Türü  İstatistiksel Gösterge  Endeks-Değişim       İktisadi Faaliyet                                     Dış Ticaret Endeks Sektör Kodları  Temel Dönem/Yıl  Coğrafi Kapsam  Düzeltme           
2015     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                                                     0+1 Gıda, içecek ve tütün          2015             Türkiye         Uygulanabilir değil    100.0
2016     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                                                     0+1 Gıda, içecek ve tütün          2015             Türkiye         Uygulanabilir değil     95.5
2017     Yıllık          Oran         Altın Hariç Dış Ticaret Hadleri  Diğer             Dış Ticaret Endeksleri  Uygulanabilir değil  -                   